In [1]:
from pathlib import Path
from datetime import datetime
Path("./exports").mkdir(exist_ok=True)

datetime.now().strftime("%Y%m%d_%H%M%S")

'20261109_105013'

In [4]:
from dataclasses import dataclass
from typing import Literal
from model import nividia_model
from pydantic_ai.agent import Agent
from pydantic_ai import RunContext
from datetime import datetime

model = nividia_model()


@dataclass
class ClientDeps:
    name: str
    goal: str
    # defaults -> optional
    experience_lvl: Literal["beginner", "intermediate", "advanced"] = "beginner"
    weekly_hours: int = 8
    current_role: str | None = None


# specialist agents
study_agent = Agent(
    model=model,
    deps_type=ClientDeps,
    instructions=(
        "You are a study expert. Give concrete study plans and learning techniques."
    ),
)

career_agent = Agent(
    model=model,
    deps_type=ClientDeps,
    instructions="You are a career advisor. Give practical advice on jobs, resumes, and career paths.",
)

coach_agent = Agent(
    model=model,
    deps_type=ClientDeps,
    instructions=(
        "You are a personal coach. Follow these steps IN ORDER:\n"
        "1. Call `ask_study_agent` for study questions.\n"
        "2. Call `ask_career_agent` for career questions.\n"
        "3. Combine the answers into ONE short, simple response.\n"
        "4. Call `export_report` with that final response as `text`.\n"
        "5. Reply to the user with the final response."
    ),
)


@study_agent.instructions
def study_name(ctx: RunContext[ClientDeps]) -> str:
    return f"Student's name is {ctx.deps.name}. Address them by name"


@career_agent.instructions
def career_name(ctx: RunContext[ClientDeps]) -> str:
    return f"Person you are advising is {ctx.deps.name}. Address them by name"


@coach_agent.instructions
def coach_name(ctx: RunContext[ClientDeps]) -> str:
    return f"You are coaching {ctx.deps.name}. Address them by name"


# delegation tool
@coach_agent.tool
async def ask_study_agent(ctx: RunContext[ClientDeps], question: str) -> str:
    """ask study specialist for advice on learning and studying"""
    result = await study_agent.run(question, deps=ctx.deps, usage=ctx.usage)
    return result.output


@coach_agent.tool
async def ask_career_agent(ctx: RunContext[ClientDeps], question: str) -> str:
    """ask career specialist for advice on what to choose for career"""
    result = await career_agent.run(question, deps=ctx.deps, usage=ctx.usage)
    return result.output


@coach_agent.tool
def export_report(ctx: RunContext[ClientDeps], text: str):
    """exports the text to specified path"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    exports_dir = Path("exports")
    exports_dir.mkdir(exist_ok=True)
    path = exports_dir / f"{ctx.deps.name.casefold()}_{timestamp}.md"
    path.write_text(text)


result = await coach_agent.run(
    "I'm learning Python and want to become a data engineer. What should I do?",
    deps=ClientDeps(
        name="Martin", goal="Looking for studying opportunities or work opportunities"
    ),
)

result

AgentRunResult(output='Hi Martin! To become a data engineer while learning Python:\n\n1. **Master Python for data** (not web dev): data structures, file I/O, APIs, error handling, logging, virtual environments.\n2. **Learn SQL deeply**—joins, CTEs, window functions, indexing, query optimization (often more important than Python for DE roles).\n3. **Add data tools**: Pandas for manipulation, PySpark for big data, dbt for transformations, and Airflow/Prefect for orchestration.\n4. **Pick one cloud platform** (AWS, GCP, or Azure) and learn its core data services (storage, warehouse, compute).\n5. **Build 3 portfolio projects** on GitHub: an API-to-database pipeline, a dbt modeling project, and a cloud-based ETL pipeline.\n6. **Use Git**, document your work with READMEs, and quantify results on your resume.\n\n**Start this week**: Write a Python script that pulls data from a public API and saves it to a CSV or SQLite database, then push it to GitHub.')